In [1]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, make_scorer, precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

In [2]:
# Project paths and settings
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.transformers import DropColumns, IQRClipper, ScreenTimeConsistency

DATA_PATH = PROJECT_ROOT / "data/raw/student_mental_health.csv"
MODEL_PATH = PROJECT_ROOT / "models/mental_health_risk_pipeline.joblib"

RANDOM_STATE = 42
CLASS_NAMES = ["Low", "Medium", "High"]
sns.set_theme(style="whitegrid")

Data split

In [3]:
df = pd.read_csv(DATA_PATH)

X = df.drop(columns="Mental_Health_Risk")
y = df["Mental_Health_Risk"].map({"Low": 0, "Medium": 1, "High": 2})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (1600, 16)
Testing set: (400, 16)


preprocessing pipeline

In [4]:
numeric_cols = [
    "Age",
    "Year_of_Study",
    "Sleep_Hours",
    "Study_Hours_Per_Week",
    "GPA",
    "Social_Media_Hours",
    "Screen_Time_Hours",
    "Financial_Stress",
    "Family_Support",
    "Attendance_Rate",
]

ordinal_cols = [
    "Physical_Activity",
    "Diet_Quality",
    "Part_Time_Job",
    "Previous_Mental_Health_History",
]

nominal_cols = ["Gender"]

clip_cols = [
    "Sleep_Hours",
    "Study_Hours_Per_Week",
    "GPA",
    "Social_Media_Hours",
    "Screen_Time_Hours",
    "Attendance_Rate",
]

In [5]:
def create_preprocessor():
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    ordinal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[
            ["Low", "Moderate", "High"],
            ["Poor", "Average", "Good"],
            ["No", "Yes"],
            ["No", "Yes"],
        ])),
    ])

    nominal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("numerical", numerical_pipeline, numeric_cols),
        ("ordinal", ordinal_pipeline, ordinal_cols),
        ("nominal", nominal_pipeline, nominal_cols),
    ])


def create_pipeline(model):
    return Pipeline([
        ("drop_identifier", DropColumns(["Student_ID"])),
        ("screen_time_check", ScreenTimeConsistency()),
        ("outlier_clipper", IQRClipper(clip_cols)),
        ("preprocessor", create_preprocessor()),
        ("model", model),
    ])

Cross validation and tuning